# 02 Report: tables and figures from runs.jsonl

Every number for the paper is produced here and written to `MyDrive/paper1/report/` (CSV at full precision, plus LaTeX).

In [ ]:
# Setup: mount Drive and import the project code.
# Put the project folder (src/, configs/) at MyDrive/paper1/code, or set REPO_URL to a git remote.
from google.colab import drive
drive.mount("/content/drive")

import logging, subprocess, sys
REPO_URL = ""  # optional
CODE = "/content/drive/MyDrive/paper1/code"
if REPO_URL:
    CODE = "/content/zdssl"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=False)
    subprocess.run(["git", "-C", CODE, "pull"], check=True)
sys.path.insert(0, f"{CODE}/src")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

from zdssl.config import config_hash, load_config, resolve_paths
cfg = load_config(f"{CODE}/configs/paper1.yaml")
paths = resolve_paths(cfg)
print("code:", CODE, "| results:", paths["runs"])


In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import display
from zdssl.report import flatten, load_runs, write_report, budget_trajectory, profile_bot_flows
from zdssl.runner import build_matrix, completed_keys, scores_path
pd.set_option("display.max_columns", 40, "display.width", 250, "display.float_format", "{:.4f}".format)

runs = load_runs(paths["runs"], config_hash(cfg))  # runs made with any other config are left out
flat = flatten(runs)
done = {r["key"] for r in runs}
for stage in ("p1", "p2", "ablation", "condA"):
    specs = build_matrix(cfg, stage)
    print(f"{stage:9s} {sum(s.key in done for s in specs):3d} / {len(specs)} runs present")
print("devices:", sorted(flat["device"].unique()), "| config hashes:", sorted({r["config_hash"] for r in runs}))


In [ ]:
# All tables. Check n_seeds in each: a value below 10 means that row is not complete yet.
tables = write_report(flat, cfg, paths["report_dir"])
for name, table in tables.items():
    print("\n##", name); display(table)


In [ ]:
# Figures 2 and 3 (Fig. 3 uses the pilot seed, 10% budget, decoupled score).
from zdssl.dataset import load_dataset
from zdssl.figures import fig2_budget_trajectory, fig3_operating
fig2_budget_trajectory(tables["fig2_budget_trajectory"], paths["report_dir"] / "fig2_budget_trajectory.pdf")

seed, budget = cfg["matrix"]["pilot_seed"], cfg["matrix"]["primary_budget"]
case = next(r for r in runs if r["key"] == f"S4_E06|F|B|{budget:g}|{seed}")
scores = np.load(scores_path(paths["scores_dir"], case["key"]))["decoupled_test"]
ds = load_dataset(cfg)
taus = {k: v["threshold"] for k, v in case["zero_day"]["decoupled"]["operating_points"].items()}
fig3_operating(scores, ds.test_is_zero_day, taus, paths["report_dir"] / "fig3_operating.pdf",
               title_suffix=f" (seed {seed})")
print("AUROC", case["zero_day"]["decoupled"]["auroc"], taus)


In [ ]:
# Zero-day flow profiling: long sessions vs short keepalive flows.
# This is a description of the test flows, not model tuning. Look at the distribution, pick the cut,
# and state the rule in the paper.
from zdssl.splits import load_splits
frame, manifest = load_splits(paths["splits"], paths["manifest"])
test_frame = frame.loc[frame["split"] == "test"]
bot = test_frame.loc[test_frame[cfg["data"]["label_column"]] == cfg["data"]["zero_day_label"]]
display(bot[["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"]].describe(percentiles=[.5, .9, .95, .99]))

PROFILE_COLUMN, PROFILE_CUT = "Flow Duration", 1_000_000   # microseconds; edit after looking above
profile = profile_bot_flows(test_frame, ds.test_is_zero_day, scores, taus["benign"], PROFILE_COLUMN, PROFILE_CUT)
profile.to_csv(paths["report_dir"] / "bot_flow_profile.csv", index=False)
profile


In [ ]:
# Protocol numbers for Section 4.1, Fig. 1 and the parameter counts.
ingest = json.loads(paths["ingest_stats"].read_text())["summary"]
case_params = case["params"]
numbers = {
    "flows_after_cleaning": ingest["clean_rows"], "repeated_header_lines": ingest["header_rows"],
    "zero_day_flows_in_dataset": ingest["label_counts"][cfg["data"]["zero_day_label"]],
    "split_sizes": manifest["split_sizes"], "n_features_B": manifest["n_features_B"],
    "calibration_benign_flows": manifest["split_class_counts"]["tune"][cfg["data"]["benign_label"]],
    "quarantine": manifest["quarantine"],
    "validation_rows_sharing_a_train_feature_profile": manifest["validation_rows_sharing_a_train_feature_profile"],
    "budget_train_sizes": {f"{b:g}": int(flat.loc[np.isclose(flat["budget"], b) & (flat["model"] == "RF")
                                                  & (flat["condition"] == "B"), "n_train"].median())
                           for b in sorted(flat["budget"].unique(), reverse=True)},
    "parameters": case_params,
}
(paths["report_dir"] / "protocol_numbers.json").write_text(json.dumps(numbers, indent=2))
numbers
